# 05. 결과 리포트
핵심 질문: ① 크기가 커지면 꼬리(q0.05) 캘리브레이션이 좋아지는가? ② ARIMA 대비 위기 직전·직후 반응은?

`python run.py evaluate` 로 요약 CSV 생성 후 실행.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src.common import METRICS
g = pd.read_csv(METRICS/'summary_groups.csv'); t = pd.read_csv(METRICS/'summary_tau.csv')
g.query("regime=='all'").sort_values(['series','horizon','ticker','avg_pinball']).head(40)

In [ ]:
# MDD q0.05 꼬리: 위반율(목표 5%), Kupiec/Christoffersen p-value (표본이 작으니 해석 주의)
t.query("series in ['m5','m20'] and tau==0.05 and regime=='all'")[['model','ticker','series','horizon','ctx','n','hits','violation_rate','pinball','kupiec_p','christoffersen_p']].sort_values(['series','ticker','pinball'])

In [ ]:
# 위기 vs 평시
t.query("series=='m20' and tau==0.05").pivot_table(index=['ticker','model','ctx'], columns='regime', values='violation_rate')

In [ ]:
from src.viz.plots import size_comparison, calibration_plot
size_comparison(t, 'm20', 20, 0.05); calibration_plot(t, 'KOSPI', 'm20', 20)

In [ ]:
# 밴드 해석: 실현 MDD 가 사전 예측 밴드의 어디에 있었는가
from src.eval.summarize import load_all_forecasts
from src.eval.bands import classify_mdd
fc = load_all_forecasts()
x = fc.query("model=='timestral-209M' and series=='m5' and ticker=='KOSPI' and ctx==1024").copy()
x['state'] = classify_mdd(x)
x['state'].value_counts(normalize=True)